# UMAP: gene expression & cell annotations

Visualize cells in 2-D UMAP space, colored by (1) gene expression or (2) a categorical obs column.

In [ ]:
try:
    import micropip
    await micropip.install(['antclient', 'pandas', 'matplotlib'])
except:
    pass  # already installed in standard Python
print('ready')

In [ ]:
from antclient import AnthiveClient
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

try:
    import js
    api_url = f"{js.location.origin}/api/"
except:
    api_url = None

client = AnthiveClient(api_url)
print(f'Connected to {client.base_url}')

## Configuration

Set your database, embedding name, gene, and grouping column below.

In [ ]:
db_id          = 'SF9/SAN'   # use client.get_databases() to list options
embedding_name = 'Umap'      # use info['embeddings'] to list available
gene           = 'Apoe'      # gene to visualize
cat_col        = 'genotype'  # categorical obs column

info = client.get_database_info(db_id)
layer = info.get('default_layer')
if layer is None:
    raise ValueError(f"{db_id} has no default_layer — pick a different dataset or set it with ant-duckdb-check")

print(f'Database  : {db_id}')
print(f'Cells     : {info["n_cells"]:,}')
print(f'Layer     : {layer}')
print(f'Embeddings: {info["embeddings"]}')

In [ ]:
# Load UMAP coordinates
umap_df = client.get_embedding_data(db_id, embedding_name, format='dataframe')

# Load gene expression and categorical metadata (single server call each)
expr_df = client.get_cells(db_id, genes=[gene], layer=layer, format='dataframe')
cat_df  = client.get_cells(db_id, metadata=[cat_col], format='dataframe')

# Merge on cell_name
df = umap_df.merge(expr_df[['cell_name', gene]], on='cell_name')
df = df.merge(cat_df[['cell_name', cat_col]], on='cell_name')

print(f'Loaded {len(df):,} cells')
df.head(3)

## 1. UMAP colored by gene expression

Continuous colormap — dark = low, bright = high.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    df['dim_0'], df['dim_1'],
    c=df[gene], s=1, alpha=0.5, cmap='viridis', rasterized=True
)
plt.colorbar(sc, ax=ax, label=f'{gene}')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title(f'{gene} — {db_id}')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()

## 2. UMAP colored by categorical column

Each category gets a distinct color.

In [ ]:
categories = sorted(df[cat_col].dropna().unique())
palette = list(mcolors.TABLEAU_COLORS.values())
colors = {cat: palette[i % len(palette)] for i, cat in enumerate(categories)}

fig, ax = plt.subplots(figsize=(7, 5))
for cat, grp in df.groupby(cat_col):
    ax.scatter(grp['dim_0'], grp['dim_1'],
               s=1, alpha=0.5, color=colors[cat], label=str(cat), rasterized=True)

ax.legend(markerscale=6, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title(f'{cat_col} — {db_id}')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()